# 04 · Delta Lake 101
*Every write is all-or-nothing, so a half-written report never reaches an analyst.*

Coulson's bronze tables hold whatever arrived. Before the data moves on, you learn what the table format underneath actually promises.
You copy the agent roster into a small table and then poke at it: reject a bad write, add a column, upsert promotions and recruits,
read the commit log, and tidy up the files.

| Concept | Where you meet it |
|---|---|
| A Delta table is data files plus a transaction log; every write is one atomic commit | the next section, then every step |
| Schema enforcement: a write that does not match the table is rejected | Step 2 |
| Schema evolution: changing the schema on purpose | Step 3 |
| `MERGE`: update the rows that match, insert the rest | Step 4 |
| Table history: one version per commit | Step 5 |
| Small files and `OPTIMIZE` | Steps 6 and 7 |
| `VACUUM` and the retention period | Step 8 |

Deep dive: [docs/04-delta-lake-101.md](../../docs/04-delta-lake-101.md)

**Needs:** chapter 03 ran (`shield.bronze.agents`, 50 rows) and the `shield.silver` schema from chapter 01. You need `CREATE TABLE` on `shield.silver`; any compute works, serverless included.
The **Expected** numbers assume the dataset from chapter 02 (50 agents, seed 42); other data gives other numbers and a ❌, which is not a bug.
Nothing here has been run in a workspace yet, so every **Expected** is an expectation, not a recorded result.
Run the steps in order. Step 1 resets `shield.silver.agents`, so running the whole notebook again always starts clean; each later step says whether it is safe to run on its own again.

## Delta Lake: data files plus a log

**The idea:** a Delta table is a folder of ordinary Parquet data files plus a transaction log (`_delta_log`) that lists which files make up the table right now.
Every write is one commit appended to that log, so it is all-or-nothing (this is what ACID means here): a reader sees the table before a commit or after it, never half of it.
Delta is a table format, not a pattern, and it is the default table format on Azure Databricks.

```
 shield.silver.agents                      (illustrative; a managed table's folder is not meant to be browsed)
 ├── part-0000-….parquet                   data files are never edited in place
 ├── part-0001-….parquet
 └── _delta_log/
      ├── …0000.json   version 0   CREATE OR REPLACE TABLE AS SELECT   + file A
      ├── …0001.json   version 1   ADD COLUMNS                         new schema
      └── …0002.json   version 2   MERGE                               − file A  + file B
```

| Because of the log | You get | Step |
|---|---|---|
| a commit is atomic | no half-written table, even if a job dies halfway | every step |
| the schema is stored in the log | writes that do not match it are rejected | Steps 2 and 3 |
| each commit is a numbered version | an audit trail, and time travel (chapter 07) | Step 5 |
| files are only added or removed from the log, never edited | `MERGE` and `OPTIMIZE` write new files; the old ones stay on storage until `VACUUM` | Steps 4 to 8 |

**How this notebook implements it**

* One small table, `shield.silver.agents`: a copy of `shield.bronze.agents` with five columns (`agent_id`, `codename`, `clearance_level`, `home_location_id`, `active`). Steps 1 to 5 change it one thing at a time.
* A second throwaway table, `shield.silver.small_files_demo`, for the file maintenance in Steps 6 to 8, so we never run maintenance experiments on a table other chapters use.
* Every step ends with a check table that reads the result from the table itself (row counts, columns, commit metrics).

**Why it helps:** you can trust a table after a failed job, you can see who changed what and when, and you get a safe way to change a schema or upsert rows.

**Not covered yet:** reading old versions and `RESTORE` (chapter 07), `CHECK` constraints (chapter 06), clustering and partitioning, and automatic maintenance (predictive optimization), which Databricks can run for managed tables so you rarely do `OPTIMIZE` and `VACUUM` by hand.

## Setup

In [ ]:
def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


from pyspark.sql import functions as F

AGENTS_TABLE = "shield.silver.agents"
DEMO_TABLE = "shield.silver.small_files_demo"
AGENT_COLUMNS = ["agent_id", "codename", "clearance_level", "home_location_id", "active"]

assert spark.catalog.tableExists("shield.bronze.agents"), "Run notebook 03 first: shield.bronze.agents is missing."
assert spark.sql("SHOW SCHEMAS IN shield LIKE 'silver'").count() == 1, "Run notebook 01 first: schema shield.silver is missing."

## 1 · A Delta table

### Step 1 · Create the table
**Goal:** a small Delta table to experiment on.

| Input | Value |
|---|---|
| `shield.bronze.agents` | 50 rows; we keep five columns and leave the lineage columns (`_source_file`, `_ingested_at`) behind |

**Expected:** `shield.silver.agents` has 50 rows and exactly those five columns, and `DESCRIBE DETAIL` reports the format `delta`.
The number of files is small (probably 1) but depends on the platform, so it is not checked.

What you will see (illustrative):

| format | numFiles | sizeInBytes |
|---|---|---|
| delta | 1 | a few KB |

`CREATE OR REPLACE TABLE` makes this step safe to re-run. It resets the table, so the column from Step 3 and the rows from Step 4 are gone afterwards, while the history of earlier runs is kept.

In [ ]:
spark.sql(
    f"""
    CREATE OR REPLACE TABLE {AGENTS_TABLE} AS
    SELECT {", ".join(AGENT_COLUMNS)} FROM shield.bronze.agents
    """
)
agents_table = spark.table(AGENTS_TABLE)
agents_detail = spark.sql(f"DESCRIBE DETAIL {AGENTS_TABLE}")
display(agents_detail.select("format", "numFiles", "sizeInBytes"))

show_checks([
    ("bronze agents", spark.table("shield.bronze.agents").count(), 50),
    ("silver agents", agents_table.count(), 50),
    ("silver columns", agents_table.columns, AGENT_COLUMNS),
    ("table format", agents_detail.first()["format"], "delta"),
])

## 2 · Schema: enforced, then changed on purpose

### Step 2 · A write that does not match is rejected
**Goal:** see **schema enforcement**: Delta compares every write with the schema stored in the log and refuses a mismatch instead of guessing.

| Input | Value |
|---|---|
| a one-row DataFrame appended to the table | `A9999 / Ghost-00 / 1 / L001 / true` plus an extra column `nickname`, which the table does not have |

**Expected:** the append raises an exception (the error text is not verified here, so the cell prints only its class) and the table has the same number of rows as before, 50 on a first run.
Nothing was written: the failed append left no commit behind. The cell is safe to re-run at any time.

In [ ]:
rows_before_write = spark.table(AGENTS_TABLE).count()
ghost_agent = spark.createDataFrame(
    [("A9999", "Ghost-00", 1, "L001", True, "the one nobody saw")],
    "agent_id string, codename string, clearance_level int, home_location_id string, active boolean, nickname string",
)

write_rejected = False
try:
    ghost_agent.write.mode("append").saveAsTable(AGENTS_TABLE)
except Exception as error:  # noqa: BLE001
    write_rejected = True
    print("Rejected:", type(error).__name__)

show_checks([
    ("append with an unknown column was rejected", write_rejected, True),
    ("rows unchanged", spark.table(AGENTS_TABLE).count(), rows_before_write),
])

### Step 3 · Add a column on purpose
**Goal:** see **schema evolution**: the schema changes only when you say so, with `ALTER TABLE ... ADD COLUMNS`. (Automatic evolution with the `mergeSchema` write option exists too; the doc covers it.)

**Expected:** `last_seen` is the sixth column with type `date`, and existing rows read `NULL` in it. The change is one new commit and rewrites no data files.

What you will see (the last rows of `DESCRIBE`):

| col_name | data_type |
|---|---|
| active | boolean |
| last_seen | date |

The cell checks for the column first, so it is safe to re-run. The last check counts rows that have a `last_seen` date: 0 on a first run. If you re-run this cell after Step 4, it shows 2 (the two merged agents), which is correct.

In [ ]:
if "last_seen" not in spark.table(AGENTS_TABLE).columns:
    spark.sql(f"ALTER TABLE {AGENTS_TABLE} ADD COLUMNS (last_seen DATE)")

agents_table = spark.table(AGENTS_TABLE)
display(spark.sql(f"DESCRIBE {AGENTS_TABLE}"))

show_checks([
    ("columns", agents_table.columns, AGENT_COLUMNS + ["last_seen"]),
    ("type of last_seen", dict(agents_table.dtypes)["last_seen"], "date"),
    ("rows with a last_seen date (first run)", agents_table.filter("last_seen IS NOT NULL").count(), 0),
])

## 3 · Upsert

### Step 4 · MERGE promotions and recruits
**Goal:** apply a batch of changes in one atomic statement. **`MERGE`** matches source rows to table rows by a key, then runs `WHEN MATCHED` on the matches (update) and `WHEN NOT MATCHED` on the rest (insert).
It is the workhorse of incremental loads. The source must have **one row per key**, otherwise the statement fails with `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`.

| Input (temp view `agent_updates`) | agent_id | codename | clearance_level | home_location_id | active |
|---|---|---|---|---|---|
| a promotion of an existing agent | A0001 | Falcon-01 | 5 | L001 | true |
| a new recruit | A9001 | Newbie-01 | 1 | L002 | true |

Both rows also get today's date in `last_seen`.

**Expected:** 51 rows afterwards (50 + the recruit). `A0001` is now `Falcon-01` with clearance 5; in the chapter 02 dataset it was `Nomad-13` with clearance 3. `A9001` exists.
Re-running is safe: the second run finds both agents and updates them, so the table stays at 51 rows. It adds one more commit to the history each time.

What you will see (the two rows):

| agent_id | codename | clearance_level | home_location_id | active | last_seen |
|---|---|---|---|---|---|
| A0001 | Falcon-01 | 5 | L001 | true | today |
| A9001 | Newbie-01 | 1 | L002 | true | today |

In [ ]:
updates = spark.createDataFrame(
    [("A0001", "Falcon-01", 5, "L001", True), ("A9001", "Newbie-01", 1, "L002", True)],
    "agent_id string, codename string, clearance_level int, home_location_id string, active boolean",
).withColumn("last_seen", F.current_date())
updates.createOrReplaceTempView("agent_updates")

spark.sql(
    f"""
    MERGE INTO {AGENTS_TABLE} AS t
    USING agent_updates AS s
    ON t.agent_id = s.agent_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """
)

merged_agents = spark.sql(f"SELECT * FROM {AGENTS_TABLE} WHERE agent_id IN ('A0001', 'A9001') ORDER BY agent_id")
display(merged_agents)

agents_by_id = {row["agent_id"]: row for row in merged_agents.collect()}
show_checks([
    ("rows after the merge", spark.table(AGENTS_TABLE).count(), 51),
    ("A0001 codename", agents_by_id["A0001"]["codename"], "Falcon-01"),
    ("A0001 clearance_level", agents_by_id["A0001"]["clearance_level"], 5),
    ("A9001 exists", "A9001" in agents_by_id, True),
])

## 4 · The transaction log

### Step 5 · Read the history
**Goal:** see that every change so far is a numbered, attributed commit. `DESCRIBE HISTORY` is the log in table form.

**Expected:** one row per commit, newest first. On a first run that is `MERGE` (version 2), `ADD COLUMNS` (1) and the `CREATE OR REPLACE` from Step 1 (0). Earlier runs add more versions below those; background maintenance on managed tables can add more.
The `MERGE` row carries metrics. A `MERGE` of our two source rows reports `numTargetRowsUpdated + numTargetRowsInserted = 2`: 1 + 1 on a first run, 2 + 0 on a re-run. The metric names come from the Databricks docs; the exact operation names in this table have not been checked in a workspace.

What you will see (illustrative):

| version | timestamp | userName | operation |
|---|---|---|---|
| 2 | … | you@… | MERGE |
| 1 | … | you@… | ADD COLUMNS |
| 0 | … | you@… | CREATE OR REPLACE TABLE AS SELECT |

The cell is read-only, so it is safe to re-run.

In [ ]:
agents_history = spark.sql(f"DESCRIBE HISTORY {AGENTS_TABLE}")
display(agents_history.select("version", "timestamp", "userName", "operation", "operationMetrics"))

latest_merge = agents_history.filter("operation = 'MERGE'").orderBy(F.desc("version")).first()
merge_metrics = latest_merge["operationMetrics"]
rows_touched = int(merge_metrics["numTargetRowsUpdated"]) + int(merge_metrics["numTargetRowsInserted"])

show_checks([
    ("commits in the history (at least 3)", agents_history.count() >= 3, True),
    ("rows touched by the latest MERGE", rows_touched, 2),
])

## 5 · Files: compact, then clean up

Every write adds files, and an update or merge adds new ones and drops the old ones from the log. Many tiny writes therefore leave many tiny files, which slows reads because each file costs a round trip.
**`OPTIMIZE`** rewrites small files into fewer, larger ones. **`VACUUM`** later deletes the files the log no longer references, once they are older than the retention period (7 days by default).
These steps use their own table, so nothing here touches `silver.agents`.

### Step 6 · Many small writes make many small files
**Goal:** create the small-files problem on purpose.

| Input | Value |
|---|---|
| `shield.silver.small_files_demo` | `(id BIGINT, v STRING)`, filled by 15 separate single-row `INSERT` statements |

**Expected:** 15 rows in about 15 files. Auto compaction and optimized writes are switched off on this table (table properties) so the effect is visible; Databricks can compact managed tables in the background, so if `numFiles` is lower than 15 that is the platform, not a bug.
The cell runs 15 statements, so it takes a minute. It recreates the table, so it is safe to re-run.

In [ ]:
spark.sql(
    f"""
    CREATE OR REPLACE TABLE {DEMO_TABLE} (id BIGINT, v STRING)
    TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'false', 'delta.autoOptimize.optimizeWrite' = 'false')
    """
)
for i in range(15):
    spark.sql(f"INSERT INTO {DEMO_TABLE} VALUES ({i}, 'x')")

files_before_optimize = spark.sql(f"DESCRIBE DETAIL {DEMO_TABLE}").first()["numFiles"]
show_checks([
    ("rows", spark.table(DEMO_TABLE).count(), 15),
    ("data files", files_before_optimize, 15),
])

### Step 7 · OPTIMIZE compacts them
**Goal:** the same 15 rows in fewer files.

**Expected:** 1 file afterwards, still 15 rows. `OPTIMIZE` is a commit like any other: the 15 old files are removed from the log (the history metrics say 15 removed, 1 added) but they are **not** deleted from storage yet.
Re-running is safe: with one file left there is nothing to compact, so the history gets no new compaction.

What you will see:

| check | result | expected | ok |
|---|---|---|---|
| data files after OPTIMIZE | 1 | 1 | ✅ |

In [ ]:
spark.sql(f"OPTIMIZE {DEMO_TABLE}")

files_after_optimize = spark.sql(f"DESCRIBE DETAIL {DEMO_TABLE}").first()["numFiles"]
optimize_commit = (
    spark.sql(f"DESCRIBE HISTORY {DEMO_TABLE}").filter("operation = 'OPTIMIZE'").orderBy(F.desc("version")).first()
)
show_checks([
    ("data files after OPTIMIZE", files_after_optimize, 1),
    ("rows unchanged", spark.table(DEMO_TABLE).count(), 15),
    ("files replaced by the OPTIMIZE commit", int(optimize_commit["operationMetrics"]["numRemovedFiles"]), files_before_optimize),
])

### Step 8 · VACUUM, as a dry run
**Goal:** see what `VACUUM` would delete, without deleting anything.

**Expected:** nothing. The 15 files from Step 7 are no longer in the table, but they became unreferenced just now and the retention period is 7 days, so `VACUUM` would keep them. The list is empty (how an empty dry run is displayed has not been checked in a workspace).
That is also the point: files younger than the retention period protect time travel and long-running readers (chapter 07). Do not shorten the retention on tables you care about.
`DRY RUN` lists up to 1,000 files and deletes nothing, so the cell is safe to re-run.

In [ ]:
vacuum_preview = spark.sql(f"VACUUM {DEMO_TABLE} DRY RUN")
display(vacuum_preview)

show_checks([("files VACUUM would delete now", vacuum_preview.count(), 0)])

## 👀 In the workspace

* **Catalog** → `shield` → `silver` → `agents` → **History** tab: the same commits as Step 5, with user and notebook.
* The **Details** tab of `small_files_demo` shows the file count, and its **History** tab shows 15 inserts followed by one `OPTIMIZE` (until you clean up).

## 🧹 Clean up

The demo table goes; `shield.silver.agents` stays, because chapter 11 applies masks and row filters to it. Re-running Step 1 replaces the table and with it those settings, so re-apply them if you had run chapter 11.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {DEMO_TABLE}")

## 🎯 Recap

* A Delta table is Parquet files plus a transaction log; every write is one atomic commit and one numbered version.
* The schema lives in the log: a mismatching write is rejected (Step 2) and a change is explicit (Step 3).
* `MERGE` upserts in a single statement, as long as the source has one row per key (Step 4).
* Many small writes make many small files; `OPTIMIZE` compacts them, and `VACUUM` removes the replaced files only after the retention period (Steps 6 to 8).

Mini-projects: [docs/04-delta-lake-101.md](../../docs/04-delta-lake-101.md#try-it-yourself)

> 💡 Next: chapter 05 builds the medallion layers (bronze, silver, gold) on top of these tables.